In [1]:
# %% Environment & Repository Cloning
# IMPORTANT: Replace 'https://github.com/nithin12342/company-cad-proposal.git' with your actual repository URL.
!git clone https://github.com/nithin12342/company-cad-proposal.git
%cd company-cad-proposal

# Mount Google Drive and setup model cache
from google.colab import drive
drive.mount('/content/drive')
import os
doctr_cache_dir = '/content/drive/MyDrive/doctr_models'
os.makedirs(doctr_cache_dir, exist_ok=True)
os.environ['HF_HOME'] = doctr_cache_dir
os.environ['HF_CACHE_DIR'] = doctr_cache_dir
print(f'Doctr models will be cached in: {doctr_cache_dir}')

# Install system dependencies
!apt-get install -y poppler-utils

# Install Python packages
!pip install -q torch torchvision opencv-python python-doctr matplotlib pydantic pdf2image google-genai
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

# Pre-download DocTR models to Drive
print('Downloading DocTR models to Drive...')
from doctr import models
fast_predictor = models.ocr_predictor(pretrained=True)
print('DocTR models downloaded successfully')

print('Environment setup complete')

Cloning into 'company-cad-proposal'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 137 (delta 50), reused 100 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 222.18 KiB | 3.27 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/company-cad-proposal
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 67.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.9/288.9 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 129.3 MB/s eta 0:00:00
   ━━━━━━━

In [2]:
# Locate drawing.pdf
import os
pdf_path = "/content/drive/MyDrive/company cad project/drawings.pdf" # Update path as needed
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF not found at {pdf_path}. Please ensure the PDF is uploaded to your Drive.")
print(f"Found drawing.pdf at: {pdf_path}")
input_pdf = pdf_path

Mounted at /content/drive
Found drawing.pdf at: /content/drive/MyDrive/company cad project/drawings.pdf


In [3]:
# 1. Fix the typo in the source file
import os
layout_file = 'src/nodes/layout.py'
if os.path.exists(layout_file):
    with open(layout_file, 'r') as f:
        content = f.read()
    # The error shows it was looking for recognition.zpredictor
    # We replace it with the correct models.ocr_predictor
    new_content = content.replace('models.recognition.zpredictor', 'models.ocr_predictor')
    with open(layout_file, 'w') as f:
        f.write(new_content)
    print(f"Fixed typo in {layout_file}")

# 2. Setup path and reload modules to ensure changes are picked up
import sys
import importlib
sys.path.insert(0, '.')

# Import the nodes
import src.nodes.triage
import src.nodes.vectorize
import src.nodes.layout
import src.nodes.dhmot

# Force reload the modified layout module
importlib.reload(src.nodes.layout)

from src.nodes.triage import PixelTriageNode
from src.nodes.vectorize import GeometricExtractionNode
from src.nodes.layout import LayoutExtractionNode
from src.nodes.dhmot import DHMoTNode

# 3. Initialize and Execute
triage_node = PixelTriageNode(node_id='node_01_triage')
vectorize_node = GeometricExtractionNode(node_id='node_02_vectorize')
layout_node = LayoutExtractionNode(node_id='node_03_layout')
dhmot_node = DHMoTNode(node_id='node_04_dhmot')

print('Running pipeline...')
triage_output = triage_node.execute(input_pdf)
if triage_output.success:
    geometry_mask_path = triage_output.data.geometry_mask_path
    text_mask_path = triage_output.data.text_mask_path
    table_mask_path = triage_output.data.table_mask_path

    vectorize_output = vectorize_node.execute(geometry_mask_path, page_number=1)
    geometry = vectorize_output.data

    layout_output = layout_node.execute(table_mask_path, text_mask_path, page_number=1)
    tables = layout_output.data

    dhmot_output = dhmot_node.execute(geometry, tables, original_img_path=geometry_mask_path)
    if dhmot_output.success:
        axiom_manifest = dhmot_output.data
        validation_overlay_path = dhmot_output.metadata.get('validation_overlay_path')
        print('Pipeline successful')
    else:
        print('DHMoT failed')
else:
    print('Triage failed')

Fixed typo in src/nodes/layout.py


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Downloading: 100%


  0%|          | 0/65814772 [00:00<?, ?it/s]

  0%|          | 0/63303144 [00:00<?, ?it/s]

Running pipeline...
Pipeline successful


In [4]:
# %%
# Display Deterministic Outputs and Download Files
import json
from IPython.display import Image, display
from google.colab import files
import os

# Print Axiom Manifest as formatted JSON
print('\n=== AXIOM MANIFEST (Pre-LLM Output) ===')
print(json.dumps(axiom_manifest, indent=4, default=str))

# Save Axiom Manifest to file for download
manifest_json = json.dumps(axiom_manifest, indent=4, default=str)
with open('axiom_manifest.json', 'w') as f:
    f.write(manifest_json)

# Display validation overlay if available
if validation_overlay_path and os.path.exists(validation_overlay_path):
    print('\n=== VALIDATION OVERLAY ===')
    try:
        display(Image(filename=validation_overlay_path))
    except Exception as e:
        print(f'Could not load overlay image: {e}')
else:
    print('\nValidation overlay path not found or file does not exist')

# Download files
print('\n=== DOWNLOADING OUTPUTS ===')
try:
    files.download('axiom_manifest.json')
    print('Downloaded: axiom_manifest.json')
except Exception as e:
    print(f'Error downloading axiom_manifest.json: {e}')

if validation_overlay_path and os.path.exists(validation_overlay_path):
    try:
        files.download(validation_overlay_path)
        print(f'Downloaded: {os.path.basename(validation_overlay_path)}')
    except Exception as e:
        print(f'Error downloading validation overlay: {e}')
else:
    print('Validation overlay not available for download.')

print('\nIsolated pre-LLM test completed successfully. Outputs are ready for download.')


=== AXIOM MANIFEST (Pre-LLM Output) ===
{
    "hyperedges": [],
    "validations": [],
    "axioms": []
}

Validation overlay path not found or file does not exist

=== DOWNLOADING OUTPUTS ===


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: axiom_manifest.json
Validation overlay not available for download.

Isolated pre-LLM test completed successfully. Outputs are ready for download.
